# EQO tour: OpenQEvo to NWQ-Sim CPU

This tour creates a typed Hamiltonian, has EQO synthesize an OpenQASM circuit with its admitted OpenQEvo bridge, then passes the resulting immutable artifact to the published NWQ-Sim CPU simulator. The handoff stays in EQO artifacts; the notebook never imports either scientific package.

**Prerequisite:** NWQ-Sim is included in the local profile. OpenQEvo is an optional, separately installed verified wheel because its upstream redistribution license is unresolved. Before running the tour, install the organization-provided wheel with `eqo local runtime install` using the exact reference and SHA-256 in the OpenQEvo capability record; verify it with `eqo local runtime list`.

This is a local, development-only CPU demonstration. It is not a QFw runtime, an HPC target, or a hardware execution path.

In [ ]:
import json
import os

from eqo import EQOClient, render_artifact, render_run

eqo = EQOClient.connect(os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080"))
eqo.health()
published = {item["id"]: item for item in eqo.workflows.list()}
tour = published.get("openqevo-nwqsim-tour")
if tour is None:
    raise RuntimeError("openqevo-nwqsim-tour is not published. Restart EQO Local.")
tour

## 1. Create a Hamiltonian artifact

The Trotter workflow's reviewed parameters are fixed in its immutable release. This artifact captures the scientific input and its checksum.

In [ ]:
hamiltonian = {
    "qubits": 2,
    "terms": [
        {"pauli": "ZI", "coefficient": 1.0},
        {"pauli": "IZ", "coefficient": 0.5},
        {"pauli": "XX", "coefficient": 0.25},
    ],
}
input_hamiltonian = eqo.artifacts.create_input(
    "qhpc.pauli-hamiltonian@1",
    json.dumps(hamiltonian, indent=2),
    name="two-qubit-evolution.json",
    labels={"example": "openqevo-nwqsim-tour"},
)
render_artifact(input_hamiltonian)

## 2. Submit the composed tour

Composer executes the published two-node workflow. Its typed edge carries the exact OpenQEvo OpenQASM artifact to NWQ-Sim; no circuit content is copied through notebook memory.

In [ ]:
run = eqo.workflows.submit(
    tour["id"],
    tour["version"],
    inputs={"hamiltonian": input_hamiltonian.id},
)
completed = run.wait(timeout=360)
if completed.state != "succeeded":
    raise RuntimeError(f"OpenQEvo to NWQ-Sim tour ended in {completed.state}; inspect render_run(completed).")
render_run(completed)
generated_circuit = completed.artifacts.by_type("qhpc.quantum-circuit@1")
render_artifact(generated_circuit)
print(generated_circuit.read_text())
render_artifact(completed.artifacts.by_type("qhpc.evolution-synthesis-report@1"))
render_artifact(completed.artifacts.by_type("qhpc.nwqsim-measurement-counts@1"))